In [1]:
!pip install iterative-stratification tqdm scikit-learn numpy torch transformers pandas scikit-multilear datetime

ERROR: Could not find a version that satisfies the requirement scikit-multilear (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for scikit-multilear


In [2]:
import datetime
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F

start_time = datetime.datetime.now()
print(f"Script started at {start_time}")

timing_info = {}
timing_info['Start timing'] = start_time

/Users/carlotacatot/Docs/pHd/EmotionClassification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Script started at 2026-04-07 15:46:13.680731


In [43]:
# Configuration parameters
two_emotions = False
file_name = "GroundTruth.csv"
checkpoint = "bert-base-cased"
num_folds = 5
emotions = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral"]

# Learning constraints
max_labels = 3

if two_emotions:
    file_name = "GroundTruthJoySadness.csv"
    emotions = ["Joy", "Sadness"]

# Undersampling configuration
undersample = True
cut_off = 150

# Class weight configuration
class_weight = True
function = 0  # options: BCEWithLogitsLoss (0), FocalLoss (1) or CrossEntropyLoss (2)
reduction = "mean" # options: mean or sum
gamma = 2.0  # for Focal Loss, when incrementing the value, focus more on hard examples.

In [44]:
# Configure output file path
suffixes = []
if undersample:
    suffixes.append("undersample")
if class_weight:
    suffixes.append("class_weight")
    if function == 0:
        suffixes.append("bce_with_logits_loss")
    elif function == 1:
        suffixes.append("focal_loss")
    elif function == 2:
        suffixes.append("cross_entropy_loss")

    suffixes.append(f"reduction_{reduction}")

prefix = "results_2e" if two_emotions else "results_all"

suffix_part = f"_{'_'.join(suffixes)}" if suffixes else ""

# Combine everything
output_path_csv = f"results/{prefix}_max{max_labels}_with_folds_{num_folds}{suffix_part}.csv"

In [45]:
df = pd.read_csv(file_name, quotechar='"', sep=';')
#df = df[df['Reject'] != 1]  # Filter out rows where Reject is 1

In [46]:
if undersample:
    start_undersampling = datetime.datetime.now()
    rng = np.random.default_rng(42)
    df = df.reset_index(drop=True)
    pos_counts = df[emotions].sum(axis=0).astype(int)
    targets = np.minimum(cut_off, pos_counts).astype(int)

    selected_idx = set()
    for lbl, tgt in zip(emotions, targets):
        if tgt <= 0:
            continue
        pos_idx = df.index[df[lbl] == 1].to_numpy()
        if len(pos_idx) <= tgt:
            chosen = pos_idx
        else:
            chosen = rng.choice(pos_idx, size=int(tgt), replace=False)
        selected_idx.update(int(x) for x in chosen)

    undersampled = df.loc[sorted(selected_idx)].reset_index(drop=True)
    print("Target per label:", dict(zip(emotions, targets.tolist())))
    print("Result per-label counts:\n", undersampled[emotions].sum().to_dict())
    print("Total rows selected:", len(undersampled))
    df = undersampled
    end_undersampling = datetime.datetime.now()
    timing_info['Undersampling time'] = end_undersampling - start_undersampling

Target per label: {'Joy': 150, 'Trust': 150, 'Fear': 24, 'Surprise': 54, 'Sadness': 150, 'Disgust': 50, 'Anger': 27, 'Anticipation': 150, 'Neutral': 88}
Result per-label counts:
 {'Joy': 191, 'Trust': 150, 'Fear': 24, 'Surprise': 54, 'Sadness': 180, 'Disgust': 50, 'Anger': 27, 'Anticipation': 155, 'Neutral': 88}
Total rows selected: 773


In [47]:
X = df[['sentence']].values
y = df[emotions].values

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [48]:
pos_weight = None

if class_weight:
    start_time_class_weight = datetime.datetime.now()
    num_pos = df[emotions].sum().values
    num_neg = len(df) - num_pos

    weights = num_neg / (num_pos + 1e-6)
    pos_weight = torch.tensor(weights, dtype=torch.float).to(device)

    print("Label counts (Positives):\n", num_pos)
    print("Computed pos_weights (Neg/Pos):\n", pos_weight)

    print("Weights per emotion:")
    for emotion, w in zip(emotions, pos_weight.tolist()):
        print(f"{emotion:12s}: {w:.3f}")

    timing_info['Class weight calculation time'] = datetime.datetime.now() - start_time_class_weight

Label counts (Positives):
 [191 150  24  54 180  50  27 155  88]
Computed pos_weights (Neg/Pos):
 tensor([ 3.0471,  4.1533, 31.2083, 13.3148,  3.2944, 14.4600, 27.6296,  3.9871,
         7.7841])
Weights per emotion:
Joy         : 3.047
Trust       : 4.153
Fear        : 31.208
Surprise    : 13.315
Sadness     : 3.294
Disgust     : 14.460
Anger       : 27.630
Anticipation: 3.987
Neutral     : 7.784


In [49]:
class MultiLabelFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None, reduction='mean'):
        super(MultiLabelFocalLoss, self).__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs,
            targets,
            reduction=reduction,
            pos_weight=self.pos_weight
        )

        pt = torch.exp(-bce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [50]:
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized

In [51]:
def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)
    return batch_inputs

In [52]:
def create_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(emotions),
        problem_type="multi_label_classification"
    )
    model.to(device)
    return model

In [53]:
def get_constrained_predictions(logits):
    """
    Get top-k predictions based on max_labels constraint.

    Args:
        logits: tensor of shape (batch_size, num_labels)

    Returns:
        numpy array of shape (batch_size, num_labels) with 0/1 predictions
    """
    probs = torch.sigmoid(logits)

    # Get top max_labels indices per sample
    num_labels = probs.shape[1]
    k = min(max_labels, num_labels)

    _, top_indices = torch.topk(probs, k=k, dim=1)

    final_preds = torch.zeros_like(probs, dtype=torch.int)

    # Set predicted labels to 1
    for i, indices in enumerate(top_indices):
        final_preds[i, indices] = 1

    return final_preds.numpy()

In [54]:
multilabel_stratified_in_folds = MultilabelStratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
all_metrics = []
fold_results = []

In [55]:
for fold, (train_idx, val_idx) in enumerate(multilabel_stratified_in_folds.split(X, y)):
    timing_fold = datetime.datetime.now()
    print(f"\n===== Fold {fold + 1} =====")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = EmotionDataset(train_df, tokenizer, emotions)
    val_dataset = EmotionDataset(val_df, tokenizer, emotions)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

    model = create_model()

    optimizer = AdamW(model.parameters(), lr=2e-5)
    num_epochs = 3
    num_training_steps = num_epochs * len(train_loader)
    lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    # ---- training loop ----
    model.train()
    progress_bar = tqdm(range(num_training_steps))

    if class_weight and function == 0:
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=reduction)
    elif class_weight and function == 1:
        loss_fct = MultiLabelFocalLoss(gamma=2.0, pos_weight=pos_weight, reduction=reduction)
    elif class_weight and function == 2:
        loss_fct = torch.nn.CrossEntropyLoss(weight=pos_weight, reduction=reduction)

    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            if class_weight:
                logits = outputs.logits
                loss = loss_fct(logits, batch["labels"].float())
            else:
                loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

    # ---- evaluation ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch["labels"].cpu().numpy().astype(int)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            outputs = model(**inputs)
            logits = outputs.logits.cpu()
            preds = get_constrained_predictions(logits)
            all_preds.append(preds)
            all_labels.append(labels)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    sentences = val_df["sentence"].values


    def binvec_to_names(vec):
        names = [emotions[i] for i, v in enumerate(vec) if int(v) == 1]
        return ", ".join(names) if names else "None"


    true_emotions = [binvec_to_names(row) for row in y_true]
    pred_emotions = [binvec_to_names(row) for row in y_pred]

    df_predictions = pd.DataFrame({
        "sentence": sentences,
        "true_emotions": true_emotions,
        "pred_emotions": pred_emotions
    })

    # show a quick preview and optionally save per fold
    #print(df_predictions.head(20).to_string(index=False))
    df_predictions.to_csv(f"predictions/predictions_fold_{fold + 1}_out_of_{num_folds}.csv", index=False)

    fold_results.append({
        "fold": fold + 1,
        "y_true": y_true,
        "y_pred": y_pred
    })

    metrics = {
        "Subset Accuracy (Exact Match)": accuracy_score(y_true, y_pred),
        "Micro Precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro Recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro F1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Weighted Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

    # --- Print the Summary ---
    print(f"\n###### Metrics for fold {fold + 1} ######")
    for k, v in metrics.items():
        print(f"{k:25s}: {v:.4f}")

    all_metrics.append(metrics)
    end_timing_fold = datetime.datetime.now()
    timing_info[f'Fold {fold + 1} time'] = end_timing_fold - timing_fold


===== Fold 1 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 234/234 [05:13<00:00,  1.76s/it]


###### Metrics for fold 1 ######
Subset Accuracy (Exact Match): 0.0000
Micro Precision          : 0.3224
Micro Recall             : 0.7989
Micro F1                 : 0.4594
Macro Precision          : 0.2880
Macro Recall             : 0.7349
Macro F1                 : 0.4014
Weighted Precision       : 0.3600
Weighted Recall          : 0.7989
Weighted F1              : 0.4854

===== Fold 2 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 234/234 [05:17<00:00,  1.36s/it]

100%|██████████| 231/231 [05:34<00:00,  1.13s/it]


###### Metrics for fold 2 ######
Subset Accuracy (Exact Match): 0.0000
Micro Precision          : 0.3043
Micro Recall             : 0.8033
Micro F1                 : 0.4414
Macro Precision          : 0.2681
Macro Recall             : 0.6969
Macro F1                 : 0.3762
Weighted Precision       : 0.3494
Weighted Recall          : 0.8033
Weighted F1              : 0.4731

===== Fold 3 =====


...


###### Metrics for fold 3 ######
Subset Accuracy (Exact Match): 0.0065
Micro Precision          : 0.3052
Micro Recall             : 0.7622
Micro F1                 : 0.4359
Macro Precision          : 0.2687
Macro Recall             : 0.7129
Macro F1                 : 0.3822
Weighted Precision       : 0.3356
Weighted Recall          : 0.7622
Weighted F1              : 0.4601

===== Fold 4 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 234/234 [05:25<00:00,  1.39s/it]

100%|██████████| 234/234 [04:36<00:00,  1.04it/s]


###### Metrics for fold 4 ######
Subset Accuracy (Exact Match): 0.0000
Micro Precision          : 0.2778
Micro Recall             : 0.7104
Micro F1                 : 0.3994
Macro Precision          : 0.2430
Macro Recall             : 0.6212
Macro F1                 : 0.3379
Weighted Precision       : 0.3263
Weighted Recall          : 0.7104
Weighted F1              : 0.4379

===== Fold 5 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 234/234 [05:01<00:00,  1.16it/s]


###### Metrics for fold 5 ######
Subset Accuracy (Exact Match): 0.0067
Micro Precision          : 0.2867
Micro Recall             : 0.7011
Micro F1                 : 0.4069
Macro Precision          : 0.2390
Macro Recall             : 0.6355
Macro F1                 : 0.3342
Weighted Precision       : 0.2988
Weighted Recall          : 0.7011
Weighted F1              : 0.4137


In [56]:
print("\n###### Summary per fold ######")
timing_results = datetime.datetime.now()

df_metrics = pd.DataFrame(all_metrics).T  # transpose to have metrics as rows
df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
print("\n###### Overall Metrics per Fold ######")
print(df_metrics.to_markdown(floatfmt=".4f"))


###### Summary per fold ######

###### Overall Metrics per Fold ######
|                               |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:------------------------------|---------:|---------:|---------:|---------:|---------:|
| Subset Accuracy (Exact Match) |   0.0000 |   0.0000 |   0.0065 |   0.0000 |   0.0067 |
| Micro Precision               |   0.3224 |   0.3043 |   0.3052 |   0.2778 |   0.2867 |
| Micro Recall                  |   0.7989 |   0.8033 |   0.7622 |   0.7104 |   0.7011 |
| Micro F1                      |   0.4594 |   0.4414 |   0.4359 |   0.3994 |   0.4069 |
| Macro Precision               |   0.2880 |   0.2681 |   0.2687 |   0.2430 |   0.2390 |
| Macro Recall                  |   0.7349 |   0.6969 |   0.7129 |   0.6212 |   0.6355 |
| Macro F1                      |   0.4014 |   0.3762 |   0.3822 |   0.3379 |   0.3342 |
| Weighted Precision            |   0.3600 |   0.3494 |   0.3356 |   0.3263 |   0.2988 |
| Weighted Recall               |   0.

In [57]:
emotion_metrics = {emotion: {"Precision": [], "Recall": [], "F1": [], "Accuracy": []} for emotion in emotions}

for fold_data in fold_results:
    y_true = fold_data["y_true"]
    y_pred = fold_data["y_pred"]
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    acc = [accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]

    for i, emotion in enumerate(emotions):
        emotion_metrics[emotion]["Precision"].append(prec[i])
        emotion_metrics[emotion]["Recall"].append(rec[i])
        emotion_metrics[emotion]["F1"].append(f1[i])
        emotion_metrics[emotion]["Accuracy"].append(acc[i])

In [58]:
print("\n###### Per Emotion Metrics per Fold ######")
for emotion, metrics_dict in emotion_metrics.items():
    df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
    print(f"\n--- {emotion} ---")
    print(df_emotion.to_markdown(floatfmt=".4f"))


###### Per Emotion Metrics per Fold ######

--- Joy ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|---------:|---------:|---------:|---------:|---------:|
| Precision |   0.3846 |   0.3462 |   0.4048 |   0.4516 |   0.3333 |
| Recall    |   0.7895 |   0.9474 |   0.8718 |   0.7368 |   0.8421 |
| F1        |   0.5172 |   0.5070 |   0.5528 |   0.5600 |   0.4776 |
| Accuracy  |   0.6316 |   0.5652 |   0.6429 |   0.7179 |   0.5333 |

--- Trust ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|---------:|---------:|---------:|---------:|---------:|
| Precision |   0.3867 |   0.4118 |   0.3521 |   0.3378 |   0.4576 |
| Recall    |   0.9667 |   0.9333 |   0.8333 |   0.8333 |   0.9000 |
| F1        |   0.5524 |   0.5714 |   0.4950 |   0.4808 |   0.6067 |
| Accuracy  |   0.6908 |   0.7391 |   0.6688 |   0.6538 |   0.7667 |

--- Fear ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|----

In [59]:
print("\n###### Average Per-Emotion Metrics ######")
df_emotion_avg = pd.DataFrame({
    emotion: {metric: np.mean(values) for metric, values in metrics.items()}
    for emotion, metrics in emotion_metrics.items()
})
print(df_emotion_avg.T.to_markdown(floatfmt=".4f"))


###### Average Per-Emotion Metrics ######
|              |   Precision |   Recall |     F1 |   Accuracy |
|:-------------|------------:|---------:|-------:|-----------:|
| Joy          |      0.3841 |   0.8375 | 0.5229 |     0.6182 |
| Trust        |      0.3892 |   0.8933 | 0.5413 |     0.7039 |
| Fear         |      0.1464 |   0.7100 | 0.2369 |     0.8376 |
| Surprise     |      0.1682 |   0.5200 | 0.2434 |     0.7990 |
| Sadness      |      0.4359 |   0.6889 | 0.5287 |     0.6939 |
| Disgust      |      0.1616 |   0.4800 | 0.2392 |     0.7957 |
| Anger        |      0.1056 |   0.4800 | 0.1731 |     0.8395 |
| Anticipation |      0.3536 |   0.9097 | 0.5069 |     0.6376 |
| Neutral      |      0.2075 |   0.6033 | 0.3049 |     0.6806 |


In [60]:
df_metrics = pd.DataFrame(all_metrics)
print("\n###### Cross-validation mean results ######")
print(df_metrics.mean())

timing_info[f'Printing results'] = datetime.datetime.now() - timing_results

with open(output_path_csv, mode="w", newline="") as f:
    writer = csv.writer(f, delimiter=";")

    # ===== SUMMARY PER FOLD ######
    writer.writerow(["###### Summary per fold ######"])
    df_metrics = pd.DataFrame(all_metrics).T  # metrics as rows
    df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
    df_metrics.reset_index(inplace=True)
    df_metrics.rename(columns={"index": "Metric"}, inplace=True)
    df_metrics.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
    writer.writerow([])  # blank line

    # ###### PER-EMOTION METRICS PER FOLD ######
    writer.writerow(["###### Per Emotion Metrics per Fold ######"])
    for emotion, metrics_dict in emotion_metrics.items():
        writer.writerow([f"--- {emotion} ---"])
        df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
        df_emotion.reset_index(inplace=True)
        df_emotion.rename(columns={"index": "Metric"}, inplace=True)
        df_emotion.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
        writer.writerow([])
        writer.writerow([])  # blank line

    # ###### AVERAGE PER EMOTION ######
    writer.writerow(["###### Average Per-Emotion Metrics ######"])
    df_emotion_avg = pd.DataFrame({
        emotion: {metric: np.mean(values) for metric, values in metrics.items()}
        for emotion, metrics in emotion_metrics.items()
    })
    df_emotion_avg = df_emotion_avg.T.reset_index().rename(columns={"index": "Emotion"})
    df_emotion_avg.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
    writer.writerow([])

    # ###### CROSS-VALIDATION MEAN RESULTS ######
    writer.writerow(["###### Cross-validation Mean Results ######"])
    df_metrics = pd.DataFrame(all_metrics)
    df_mean = df_metrics.mean().to_frame(name="Mean").reset_index().rename(columns={"index": "Metric"})
    df_mean.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")

    # ###### CLASS WEIGHTS ######
    if class_weight:
        writer.writerow([])
        writer.writerow(["###### Class Weights per Emotion ######"])
        writer.writerow(["Emotion", "Weight"])
        for emotion, weight in zip(emotions, pos_weight.tolist()):
            writer.writerow([emotion, f"{weight:.4f}"])
        writer.writerow([])

    writer.writerow(["###### Timings ######"])
    for k, v in timing_info.items():
        writer.writerow([k, str(v)])

    writer.writerow(["Total duration", str(datetime.datetime.now() - start_time)])

print(f"\n✅ All tables saved in one CSV file: {os.path.abspath(output_path_csv)}")


###### Cross-validation mean results ######
Subset Accuracy (Exact Match)    0.002632
Micro Precision                  0.299271
Micro Recall                     0.755165
Micro F1                         0.428600
Macro Precision                  0.261360
Macro Recall                     0.680298
Macro F1                         0.366370
Weighted Precision               0.334040
Weighted Recall                  0.755165
Weighted F1                      0.454054
dtype: float64

✅ All tables saved in one CSV file: /Users/carlotacatot/Docs/pHd/EmotionClassification/results/results_all_max3_with_folds_5_undersample_class_weight_bce_with_logits_loss_reduction_mean.csv
